# C12-classical-models — Practice p09 — Solution


The scaler and SVC fit only training rows. The manual RBF reconstruction uses the SVC's scaled support vectors and signed dual coefficients.


In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X_train_p09 = np.array([[0.0,0.0],[0.2,0.1],[-0.2,0.1],[0.1,-0.2],
                        [1.0,0.0],[-1.0,0.0],[0.0,1.0],[0.0,-1.0],
                        [0.8,0.5],[-0.8,-0.5]], dtype=np.float64)
y_train_p09 = np.array([1,1,1,1,0,0,0,0,0,0], dtype=np.int64)
X_probe_p09 = np.array([[0.0,0.0],[0.9,0.1],[-0.1,-0.1],[0.6,0.6]], dtype=np.float64)


def fit_kernel_svc(X_train, y_train, X_probe):
    if not all(isinstance(a, np.ndarray) for a in (X_train, y_train, X_probe)):
        raise ValueError("inputs must be arrays")
    if X_train.dtype != np.float64 or X_probe.dtype != np.float64 or not np.issubdtype(y_train.dtype, np.integer):
        raise ValueError("invalid dtypes")
    if X_train.ndim != 2 or X_probe.ndim != 2 or y_train.ndim != 1 or X_train.shape[0] < 2:
        raise ValueError("invalid dimensions")
    if X_train.shape[1] < 1 or X_probe.shape[1] != X_train.shape[1] or y_train.shape != (X_train.shape[0],):
        raise ValueError("shape mismatch")
    if not np.isfinite(X_train).all() or not np.isfinite(X_probe).all() or set(np.unique(y_train)) != {0, 1}:
        raise ValueError("invalid values")
    model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=4.0, gamma=1.25))
    model.fit(X_train, y_train)
    scaler, svc = model.steps[0][1], model.steps[1][1]
    scaled_probe = scaler.transform(X_probe)
    squared = ((svc.support_vectors_[:, None, :] - scaled_probe[None, :, :]) ** 2).sum(axis=2)
    kernels = np.exp(-float(svc._gamma) * squared)
    manual_scores = svc.dual_coef_[0] @ kernels + float(svc.intercept_[0])
    return {"model": model, "predictions": model.predict(X_probe),
            "scores": model.decision_function(X_probe),
            "support_indices": svc.support_.astype(np.int64, copy=True),
            "manual_scores": np.asarray(manual_scores, dtype=np.float64)}


result_p09 = fit_kernel_svc(X_train_p09, y_train_p09, X_probe_p09)


### Answer check


In [ ]:
ATOL = 1e-8
RTOL = 1e-7
assert np.allclose(result_p09["scores"], result_p09["manual_scores"], atol=ATOL, rtol=RTOL)
assert np.array_equal(result_p09["predictions"], [1, 0, 1, 0])
assert np.allclose(result_p09["scores"], [1.30756227, -0.96874703, 1.12960967, -0.85856315], atol=ATOL, rtol=RTOL)
assert result_p09["support_indices"].dtype == np.int64
support_labels_p09 = y_train_p09[result_p09["support_indices"]]
assert np.count_nonzero(support_labels_p09 == 0) >= 2 and np.count_nonzero(support_labels_p09 == 1) >= 2
